# QAOA Vanilla - MaxCut on Weighted ER Graphs

This notebook:
- Generates 5 weighted Erdős–Rényi graphs
- Runs vanilla QAOA on each graph
- Collects expectation, variance, and optimal parameters

In [1]:
from qaoa import QAOA, problems, mixers, initialstates
from qiskit_algorithms.optimizers import L_BFGS_B
import time
from src.utils import generate_er_graphs, maxcut_bruteforce

import pandas as pd
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.max_colwidth', None)

In [2]:
# Configuration
n_graphs = 1
n_nodes = 10
depth = 10   # QAOA layers (p)

In [ ]:
# Generate graphs (already weighted from your function)
graphs = generate_er_graphs(n_graphs=n_graphs, n_nodes=n_nodes)

print(f"Generated {len(graphs)} graphs")

In [ ]:
# Inspect one graph
name, G = list(graphs.items())[0]
print("Example graph:", name)
print("Nodes:", G.number_of_nodes())
print("Edges:", G.number_of_edges())

In [ ]:

def graph_to_edgelist(G):
    """Convert NetworkX graph → [[u, v, w], ...]"""
    return [[u, v, d.get("weight", 1.0)] for u, v, d in G.edges(data=True)]


def run_qaoa_on_graph(graph_name, G, depth):
    print(f"\nRunning QAOA on {graph_name}")
    start = time.time()

    # --- Initialize QAOA
    qaoa = QAOA(
        problem=problems.MaxCut(G),
        mixer=mixers.X(),
        initialstate=initialstates.Plus(),
        interpolate=True,
        optimizer=[L_BFGS_B, {
            "maxiter": 50,
            "ftol": 1e-9
        }]
    )

    # --- Step 1: Landscape (optional, only meaningful for p=1)
    if depth == 1:
        qaoa.sample_cost_landscape()

    # --- Step 2: Optimization
    qaoa.optimize(depth=depth)

    # --- Step 3: Metrics
    exp_val = qaoa.get_Exp(depth=depth)
    var_val = qaoa.get_Var(depth=depth)

    gamma = qaoa.get_gamma(depth=depth)
    beta = qaoa.get_beta(depth=depth)

    # --- Step 4: Optimal (bruteforce)
    energy_opt, best_state = maxcut_bruteforce(G)

    # --- Step 5: Approximation ratio
    approx_ratio = exp_val / energy_opt if energy_opt != 0 else None

    # --- Step 6: Edge list
    edgelist_list = graph_to_edgelist(G)

    end = time.time()

    result = {
        "graph_name": graph_name,
        "n_nodes": G.number_of_nodes(),
        "edgelist_list_len": G.number_of_edges(),
        "n_layers": depth,
        "expected_energy": exp_val,
        "variance": var_val,
        "γ_coeff": gamma,
        "β_coeff": beta,
        "approx_ratio": approx_ratio,
        "energy_mqlib": energy_opt,
        "edgelist_list": edgelist_list,
        "took_time": round(end - start, 3),
        "method": "vanilla_qaoa",
        "optimizer": "BFGS"
    }

    print(f"Expectation: {exp_val}")
    print(f"Variance: {var_val}")

    return result

In [ ]:
results = []

for graph_name, G in graphs.items():
    result = run_qaoa_on_graph(graph_name, G, depth)
    results.append(result)

In [ ]:
# for x in dir(qaoa):
#     if not x.startswith("_"):    
#         print(x)

In [ ]:
# Convert to DataFrame
df = pd.DataFrame(results)

df

In [ ]:
# Sort by best expectation (MaxCut → usually minimize energy)
df_sorted = df.sort_values(by="expected_energy")

df_sorted

In [ ]:
# Done
print("QAOA experiment complete ✅")

In [18]:
df = pd.read_csv("ADAPT.jl_results/2026-01-14_20-29/qaoa_result/qaoa_08_22_05.csv")

In [19]:
len(df)

4

In [20]:
df.tail()

,graph_name,n_nodes,edgelist_list_len,n_layers,expected_energy,variance,γ_coeff,β_coeff,approx_ratio,energy_mqlib,edgelist_list,took_time,method,optimizer,run_id
0,graph_1,8,21,8,-6.774561,1.173187,"[0.8359455505154968, 0.8401525616954356, 0.8430400624018842, 0.8438555448263476, 0.8411738804744563, 0.8337915749820549, 0.8205314428045443, 0.7989501642088213]","[3.5040014371163255, 3.4651929278061058, 3.437584394499073, 3.416558416329302, 3.396273907883566, 3.3712124736308406, 3.3366762915943315, 3.2882454336046147]",0.889050,-7.62,"[[2, 3, 0.67], [2, 4, 0.26], [2, 5, 0.24], [2, 6, 0.77], [2, 7, 0.42], [2, 8, 0.1], [3, 4, 0.46], [3, 5, 0.42], [3, 6, 0.44], [3, 7, 0.02], [1, 4, 0.43], [1, 5, 0.81], [1, 7, 0.55], [1, 8, 0.03], [4, 7, 0.1], [4, 8, 0.81], [5, 6, 0.95], [5, 7, 0.48], [5, 8, 0.84], [6, 7, 0.17], [7, 8, 0.62]]",32.694,vanilla_qaoa,BFGS,0
1,graph_2,8,12,8,-5.187363,1.014144,"[0.8239039748861978, 0.7709694198143864, 0.8367466878281233, 0.781758368972425, 0.8980229623089646, 0.9460440960422069, 0.7472573438020971, 0.8286862538613797]","[3.486133541201941, 3.562049306928027, 3.5009132984245706, 3.419050261710331, 3.319241825129879, 3.3944148354588646, 3.3244317914221764, 3.294037739054588]",0.858835,-6.04,"[[1, 2, 0.49], [1, 3, 0.82], [2, 3, 0.75], [2, 5, 0.37], [3, 5, 0.04], [3, 6, 0.32], [5, 6, 0.51], [5, 7, 0.59], [6, 4, 0.87], [4, 7, 0.83], [4, 8, 0.06], [7, 8, 0.98]]",22.945,vanilla_qaoa,BFGS,0
2,graph_1,8,21,8,-6.789961,0.944454,"[0.6242112919715374, 0.6563639686565754, 0.6864255265811513, 0.7213618671461693, 0.7619377603924923, 0.8030349730924744, 0.854292511541167, 0.9105261285511655]","[5.097421481368371, 5.080805278123805, 5.0643356870444105, 5.0403405390068015, 5.021545004425732, 4.998428634261157, 4.971622588384336, 4.9495216705246285]",0.891071,-7.62,"[[2, 3, 0.67], [2, 4, 0.26], [2, 5, 0.24], [2, 6, 0.77], [2, 7, 0.42], [2, 8, 0.1], [3, 4, 0.46], [3, 5, 0.42], [3, 6, 0.44], [3, 7, 0.02], [1, 4, 0.43], [1, 5, 0.81], [1, 7, 0.55], [1, 8, 0.03], [4, 7, 0.1], [4, 8, 0.81], [5, 6, 0.95], [5, 7, 0.48], [5, 8, 0.84], [6, 7, 0.17], [7, 8, 0.62]]",29.194,vanilla_qaoa,BFGS,1
3,graph_2,8,12,8,-5.645508,0.359606,"[0.6429309897474716, 0.6873283296955341, 0.7333492108958349, 0.7831448522851342, 0.8422566383830564, 0.9358885382345623, 1.0962691345217004, 1.3236765264999306]","[0.41650003365072596, 0.22880517758375102, 0.2514338280736265, 0.26423378840594, 0.2603460509266865, 0.2663684124422438, 0.2543587600644469, 0.14268286342962724]",0.934687,-6.04,"[[1, 2, 0.49], [1, 3, 0.82], [2, 3, 0.75], [2, 5, 0.37], [3, 5, 0.04], [3, 6, 0.32], [5, 6, 0.51], [5, 7, 0.59], [6, 4, 0.87], [4, 7, 0.83], [4, 8, 0.06], [7, 8, 0.98]]",20.876,vanilla_qaoa,BFGS,1
